In [2]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer

In [4]:

sites = pd.read_parquet('data/processed/dep_codes.parquet')
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

In [5]:
full_df = pd.read_parquet('notebooks/06_cb_regression/completo/tp_full.parquet')
presdure = pd.read_parquet('data/raw/04_PressureStatus_GTstudentproject_B.parquet')

In [7]:
presdure

,SamplingOperations_code,CodeSite_SamplingOperations,Date_SamplingOperation,Nitrogencompounds_Status1Y,Nitrogencompounds_Status180D,Nitrogencompounds_Status90D,Nitrates_Status1Y,Nitrates_Status180D,Nitrates_Status90D,Phosphorouscompounds_Status1Y,...,OrganicMatter_Status90D,SuspendedMatter_Status1Y,SuspendedMatter_Status180D,SuspendedMatter_Status90D,OrganicMicropollutants_Status1Y,OrganicMicropollutants_Status180D,OrganicMicropollutants_Status90D,MineralMicropollutants_Status1Y,MineralMicropollutants_Status180D,MineralMicropollutants_Status90D
0,S02000008_20170703,S02000008,2017-07-03,Good,Good,Good,Moderate,Moderate,Moderate,Good,...,Bad,High,High,High,Good,Good,Good,Good,Good,Good
1,S02000008_20200708,S02000008,2020-07-08,Good,Good,Good,Moderate,Moderate,Moderate,Moderate,...,Bad,High,High,High,Good,Good,Good,Good,Good,Good
2,S02000010_20070906,S02000010,2007-09-06,Good,Good,Good,Good,Good,Good,High,...,Bad,High,High,High,Moderate,Moderate,Moderate,Good,Good,Good
3,S02000010_20080811,S02000010,2008-08-11,Good,Good,Good,Good,Good,Good,High,...,Bad,High,High,High,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate
4,S02000010_20090721,S02000010,2009-07-21,Good,Good,Good,Good,Good,Good,Good,...,Bad,High,High,High,Moderate,Moderate,Moderate,Moderate,Moderate,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49226,S06710040_20090617,S06710040,2009-06-17,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,...,Unassessed,High,High,High,Good,Good,Good,Unassessed,Unassessed,Unassessed
49227,S06330110_20220913,S06330110,2022-09-13,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,...,Unassessed,Unassessed,Unassessed,Unassessed,Good,Good,Good,Unassessed,Unassessed,Unassessed
49228,S06330240_20220920,S06330240,2022-09-20,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,...,Unassessed,Unassessed,Unassessed,Unassessed,Good,Good,Good,Unassessed,Unassessed,Unassessed
49229,S06830110_20220831,S06830110,2022-08-31,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,Unassessed,...,Unassessed,Unassessed,Unassessed,Unassessed,Good,Good,Good,Unassessed,Unassessed,Unassessed


In [12]:
dates = presdure[['SamplingOperations_code','Date_SamplingOperation']]

In [13]:
full = pd.merge(full_df,dates, on = 'SamplingOperations_code',how='left')

In [14]:
full

,SamplingOperations_code,TotalAbundance_SamplingOperation,Achat02,Achca02,Achco02,Achde03,Achdr01,Acheu01,Achge01,Achla02,...,HERlvl1Code,HERlvl1Name,HERlvl2Code,Altitude,Streamsize,Uncommon_Taxons,IBD,IBD_EQR,IBD_EQR_Status,Date_SamplingOperation
0,S02000008_20170703,405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.407407,...,18,ALSACE,73,0.0,TP,0.000000,9.6,0.502924,Poor,2017-07-03
1,S02000008_20200708,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.500000,...,18,ALSACE,73,0.0,TP,5.000000,8.0,0.409357,Poor,2020-07-08
2,S02000010_20070906,400,NaN,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,...,18,ALSACE,62,246.0,None,80.000000,14.3,0.777778,Moderate,2007-09-06
3,S02000010_20090721,400,NaN,NaN,NaN,NaN,NaN,20.000000,2.5,15.000000,...,18,ALSACE,62,246.0,None,55.000000,15.0,0.818713,Good,2009-07-21
4,S02000010_20110723,398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18,ALSACE,62,246.0,None,55.276382,16.0,0.877193,Good,2011-07-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49226,S06940940_20100708,438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5,JURA-PREALPES DU NORD,2,256.0,P,2.283105,NaN,NaN,None,2010-07-08
49227,S06940940_20230623,408,NaN,NaN,NaN,NaN,NaN,4.901961,NaN,NaN,...,5,JURA-PREALPES DU NORD,2,256.0,P,19.607843,NaN,NaN,None,2023-06-23
49228,S06960950_20160629,401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5,JURA-PREALPES DU NORD,2,366.0,TP,24.937656,NaN,NaN,None,2016-06-29
49229,S06960950_20180719,416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5,JURA-PREALPES DU NORD,2,366.0,TP,33.653846,NaN,NaN,None,2018-07-19


In [3]:
path = "../../notebooks/06_cb_regression/dfs_tp_dated/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

Loaded df_1 with shape (1485, 110)
Loaded df_10 with shape (3926, 149)
Loaded df_11 with shape (1289, 164)
Loaded df_12 with shape (4677, 177)
Loaded df_13 with shape (1003, 173)
Loaded df_14 with shape (7742, 162)
Loaded df_15 with shape (1223, 161)
Loaded df_16 with shape (379, 114)
Loaded df_17 with shape (803, 152)
Loaded df_18 with shape (1164, 153)
Loaded df_19 with shape (500, 135)
Loaded df_2 with shape (352, 98)
Loaded df_20 with shape (508, 210)
Loaded df_21 with shape (2663, 185)
Loaded df_22 with shape (152, 177)
Loaded df_3 with shape (4996, 142)
Loaded df_4 with shape (596, 142)
Loaded df_5 with shape (2313, 132)
Loaded df_6 with shape (2134, 143)
Loaded df_7 with shape (524, 108)
Loaded df_8 with shape (548, 124)
Loaded df_9 with shape (10254, 170)


In [4]:

basetun = dict(
        loss_function="RMSE",
        eval_metric="RMSE",        # o "R2" si prefieres monitorear R2
        depth=5,                   # 4–6 para 2k x 200
        learning_rate=0.03,        # 0.02–0.05
        iterations=5000,           # early stopping cortará antes
        l2_leaf_reg=10,            # 8–12 reduce gap
        bagging_temperature=1.5,   # diversidad
        subsample=0.8,
        colsample_bylevel=0.7,
        random_strength=1.5,
        max_bin=128,
        random_state=42,
        verbose=0,
    )


base2 =  dict(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=3,                    # 2–4 en 
    learning_rate=0.02,         # suave
    iterations=10000,           # se cortará con ES
    l2_leaf_reg=24,             # 16–28
    min_data_in_leaf=15,        # 12–25
    bootstrap_type="Bayesian",
    bagging_temperature=2.5,    # 2.0–3.0 diversidad por pesos
    colsample_bylevel=0.45,     # 0.35–0.55 (features por árbol)
    random_strength=2.5,        # ruido en splits
    max_bin=64,                 # discretización tosca
    random_state=42,
    verbose=0
)


base22 = alt_bayes_medio = dict(
    loss_function="RMSE", eval_metric="RMSE",
    depth=2, learning_rate=0.018, iterations=12000,
    l2_leaf_reg=24, min_data_in_leaf=18,
    bootstrap_type="Bayesian", bagging_temperature=2.8,
    colsample_bylevel=0.35, random_strength=2.8,
    max_bin=48,
    sampling_frequency="PerTreeLevel",
    leaf_estimation_iterations=2,
    random_state=42, verbose=0
)

In [15]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

def train_catboost_region_tunueado(
    cleandf: pd.DataFrame,
    target: str = "IBD",
    base = dict,
    test_size: float = 0.20,
    random_state: int = 42,
    cat_params: dict | None = None,
    use_ohe: bool = False,   # False = categóricas nativas (recomendado)
) -> Tuple[CatBoostRegressor, Dict[str, Any], pd.DataFrame]:
    """
    Entrena CatBoostRegressor para una región usando cleandf (con numéricas y categóricas).
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación cat). Si use_ohe=True, tú haces OHE fuera de esta función.
    - Entrena con early stopping y regularización
    - Devuelve: modelo, métricas en holdout, y scored_df (filas sin target con predicción)
    """

    # -----------------------------
    # 1) separar train / score
    # -----------------------------
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # columnas a tirar si existen (evitar leakage)
    drop_cols = [c for c in ["IBD", "IBD_EQR", "IBD_EQR_Status"] if c in cleandf.columns]

    X = df_train.drop(columns=drop_cols + [target], errors="ignore")
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # -----------------------------
    # 2) detectar numéricas / categóricas
    # -----------------------------
    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_tr.select_dtypes(exclude=[np.number]).columns.tolist()

    # -----------------------------
    # 3) imputación ligera
    # -----------------------------
    # num: mediana no lo ghago por la estructura de taxones ; cat: literal '(missing)' (si usas OHE hazlo igual antes del OHE)
    X_tr_num = X_tr[num_cols].copy()
    X_te_num = X_te[num_cols].copy()


    X_tr_cat = X_tr[cat_cols].copy()
    X_te_cat = X_te[cat_cols].copy()
    X_tr_cat = X_tr_cat.fillna("(missing)")
    X_te_cat = X_te_cat.fillna("(missing)")

    if use_ohe:
        # -------------------------
        # 3.a) OHE (opcional)
        # -------------------------
        # Nota: si quieres OHE aquí, puedes hacerlo con pd.get_dummies para rapidez.
        # (Si ya traes OHE hecho fuera, simplemente deja use_ohe=False)
        X_tr_cat = pd.get_dummies(X_tr_cat, drop_first=False)
        X_te_cat = pd.get_dummies(X_te_cat, drop_first=False)
        # alinear columnas
        X_tr_cat, X_te_cat = X_tr_cat.align(X_te_cat, join="left", axis=1, fill_value=0)
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        cat_features = None  # ya no se usan índices de categóricas con OHE
    else:
        # -------------------------
        # 3.b) Categóricas nativas
        # -------------------------
        X_tr_proc = pd.concat([X_tr_num.reset_index(drop=True),
                               X_tr_cat.reset_index(drop=True)], axis=1)
        X_te_proc = pd.concat([X_te_num.reset_index(drop=True),
                               X_te_cat.reset_index(drop=True)], axis=1)
        # índices de columnas categóricas en el DataFrame concatenado
        # (van después de las numéricas)
        cat_features = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

    # -----------------------------
    # 4) CatBoost: base + overrides
    # -----------------------------
    base_params = base 
    if cat_params:
        base_params.update(cat_params)

    model = CatBoostRegressor(**base_params)

    # -----------------------------
    # 5) entrenar con early stopping
    # -----------------------------
    fit_kwargs = dict(
        X=X_tr_proc,
        y=y_tr,
        eval_set=(X_te_proc, y_te),
        use_best_model=True,
    )
    if not use_ohe:
        fit_kwargs["cat_features"] = cat_features  # solo si usamos categóricas nativas

    model.fit(**fit_kwargs)

    # -----------------------------
    # 6) métricas
    # -----------------------------
    pred_tr = model.predict(X_tr_proc)
    pred_te = model.predict(X_te_proc)

    r2_tr = r2_score(y_tr, pred_tr)
    r2_te = r2_score(y_te, pred_te)
    rmse_tr = mean_squared_error(y_tr, pred_tr)
    rmse_te = mean_squared_error(y_te, pred_te)
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    mae_te = mean_absolute_error(y_te, pred_te)

    metrics = {
        "R2_train": float(r2_tr),
        "R2_valid": float(r2_te),
        "MAE_train": float(mae_tr),
        "MAE_valid": float(mae_te),
        "RMSE_train": float(rmse_tr),
        "RMSE_valid": float(rmse_te),
        "best_iterations": int(model.get_best_iteration() or model.tree_count_),
        "gap": float(abs(r2_tr - r2_te)),
        "used_ohe": use_ohe,
    }

    # -----------------------------
    # 7) score para filas sin target
    # -----------------------------
    if not df_score.empty:
        Xs = df_score.drop(columns=drop_cols + [target], errors="ignore")

        Xs_num = Xs[num_cols].copy()
        Xs_cat = Xs[cat_cols].copy().fillna("(missing)")

        if use_ohe:
            Xs_cat = pd.get_dummies(Xs_cat, drop_first=False)
            # alinear con entrenamiento
            Xs_cat = Xs_cat.reindex(columns=X_tr_cat.columns, fill_value=0)

            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)
        else:
            Xs_proc = pd.concat([Xs_num.reset_index(drop=True),
                                 Xs_cat.reset_index(drop=True)], axis=1)

        df_score[target + "_pred"] = model.predict(Xs_proc)
        scored_df = df_score
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + "_pred"])

    return model, metrics, scored_df

In [6]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    if region == 2:
        base = base2
    elif region == 22:
        base=base22
    else:
        base = basetun

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region_tunueado(cleandf, target='IBD', base = base)  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['SamplingOperations_code','IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)
    print(region)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))
metrics_df

18
5
4
10
22
9
21
20
12
8
3
17
14
11
13
19
1
7
6
16
15
2


,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations,gap,used_ohe
0,18,0.996038,0.912557,0.095355,0.460648,0.016738,0.480484,4993,0.083481,False
1,5,0.995194,0.944714,0.121012,0.376820,0.029782,0.396808,4998,0.050480,False
2,4,0.998300,0.852044,0.084349,0.649936,0.012314,1.138699,4997,0.146256,False
3,10,0.988119,0.952033,0.200609,0.384953,0.097623,0.385847,4997,0.036086,False
4,22,0.982352,0.721228,0.267837,1.004248,0.131366,2.343469,11998,0.261125,False
5,9,0.973046,0.943661,0.191646,0.258801,0.088799,0.182107,4998,0.029385,False
6,21,0.995036,0.931383,0.139391,0.451474,0.034282,0.507170,4988,0.063654,False
7,20,0.997213,0.891236,0.093161,0.617662,0.017354,0.689412,4999,0.105976,False
8,12,0.987888,0.949594,0.177594,0.342793,0.065431,0.285032,4998,0.038293,False
9,8,0.992080,0.842896,0.113333,0.594339,0.033206,0.684380,4999,0.149184,False


In [7]:
predicciones

,SamplingOperations_code,IBD_pred,region
SamplingOperations_code,,,
977,S02000010_20080811,14.174752,18
978,S02000010_20100719,15.456491,18
979,S02000010_20150811,13.785765,18
980,S02000010_20160825,14.733583,18
981,S02000010_20170703,16.039005,18
...,...,...,...
347,S06700075_20120611,19.918207,2
348,S06700094_20210830,18.284056,2
349,S06700590_20210830,18.547747,2


In [11]:
predicciones.to_csv('preds_ibd_cat_pr.csv', index=False)

In [12]:
df = pd.read_csv('preds_ibd_cat_pr.csv')
df

,SamplingOperations_code,IBD_pred,region
0,S02000010_20080811,14.174752,18
1,S02000010_20100719,15.456491,18
2,S02000010_20150811,13.785765,18
3,S02000010_20160825,14.733583,18
4,S02000010_20170703,16.039005,18
...,...,...,...
5658,S06700075_20120611,19.918207,2
5659,S06700094_20210830,18.284056,2
5660,S06700590_20210830,18.547747,2
5661,S06710014_20210823,19.789796,2


In [15]:
df= pd.read_csv('pred_full_t.csv')
df

,SamplingOperations_code,IBD_pred
0,S02000010_20080811,13.923305
1,S02000010_20100719,15.618174
2,S02000010_20150811,13.511121
3,S02000010_20160825,14.599235
4,S02000010_20170703,16.000024
...,...,...
5658,S06940940_20100708,16.145701
5659,S06940940_20230623,15.413047
5660,S06960950_20160629,19.904753
5661,S06960950_20180719,19.983617


In [18]:
base_full = dict(
    loss_function="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=6.0,
    random_strength=0.8,
    bagging_temperature=0.5,  # stochastic depth/bagging         # PerTree sampling
    colsample_bylevel=0.8,    # RSM
    grow_policy="SymmetricTree",
    bootstrap_type="Bayesian",
    leaf_estimation_iterations=10,
    min_data_in_leaf=20,
    max_bin=128,              # (opcional) reduce tiempo sin perder mucho
    verbose=100,
    random_seed=42,
    od_type="Iter",
    od_wait=100,
    eval_metric="RMSE"
)

In [2]:
from catboost import CatBoostClassifier, gpu_info
print("GPUs detectadas por CatBoost:", gpu_info.get_gpu_device_count())


base_full2 = dict(
    task_type="GPU",
    devices="0",
    loss_function="RMSE",
    iterations=5000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=7.0,
    random_strength=1.0,
    bagging_temperature=0.8,
    rsm=0.8,                  # alias de colsample_bylevel en GPU
    grow_policy="Lossguide",  # mejor con muchas columnas
    min_data_in_leaf=32,
    max_bin=128,
    verbose=200,
    random_seed=42,
    od_type="Iter",
    od_wait=150,
    eval_metric="RMSE"
)


ImportError: cannot import name 'gpu_info' from 'catboost' (c:\Users\narro\AppData\Local\Programs\Python\Python313\Lib\site-packages\catboost\__init__.py)

In [19]:
model, metrics, scored_df = train_catboost_region_tunueado(
    full,
    base = base_full
)

0:	learn: 2.7350757	test: 2.7585825	best: 2.7585825 (0)	total: 201ms	remaining: 10m 2s
100:	learn: 1.1340335	test: 1.1573338	best: 1.1573338 (100)	total: 11.1s	remaining: 5m 17s
200:	learn: 0.9130863	test: 0.9463266	best: 0.9463266 (200)	total: 21.3s	remaining: 4m 57s
300:	learn: 0.8098767	test: 0.8526661	best: 0.8526661 (300)	total: 31.6s	remaining: 4m 43s
400:	learn: 0.7448457	test: 0.7947684	best: 0.7947684 (400)	total: 42.6s	remaining: 4m 36s
500:	learn: 0.6967670	test: 0.7566705	best: 0.7566705 (500)	total: 54.5s	remaining: 4m 31s
600:	learn: 0.6595987	test: 0.7249910	best: 0.7249910 (600)	total: 1m 6s	remaining: 4m 25s
700:	learn: 0.6286918	test: 0.7008972	best: 0.7008972 (700)	total: 1m 17s	remaining: 4m 14s
800:	learn: 0.6017563	test: 0.6814233	best: 0.6814233 (800)	total: 1m 28s	remaining: 4m 3s
900:	learn: 0.5787875	test: 0.6664678	best: 0.6664678 (900)	total: 1m 39s	remaining: 3m 51s
1000:	learn: 0.5581577	test: 0.6537730	best: 0.6537730 (1000)	total: 1m 50s	remaining: 3m 40

In [22]:
metrics

{'R2_train': 0.9823382285027511,
 'R2_valid': 0.9596988392067903,
 'MAE_train': 0.2454157336530481,
 'MAE_valid': 0.32988007609152536,
 'RMSE_train': 0.13731457688264395,
 'RMSE_valid': 0.31858608057166327,
 'best_iterations': 2999,
 'gap': 0.022639389295960766,
 'used_ohe': False}

In [20]:
scored_df

,SamplingOperations_code,TotalAbundance_SamplingOperation,Achat02,Achca02,Achco02,Achde03,Achdr01,Acheu01,Achge01,Achla02,...,HERlvl1Name,HERlvl2Code,Altitude,Streamsize,Uncommon_Taxons,IBD,IBD_EQR,IBD_EQR_Status,Date_SamplingOperation,IBD_pred
43568,S02000010_20080811,400,2.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ALSACE,62,246.0,None,150.000000,NaN,NaN,None,2008-08-11,13.765716
43569,S02000010_20100719,404,9.900990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ALSACE,62,246.0,None,141.089109,NaN,NaN,None,2010-07-19,15.386578
43570,S02000010_20150811,400,NaN,NaN,NaN,62.500000,NaN,NaN,NaN,5.0,...,ALSACE,62,246.0,None,45.000000,NaN,NaN,None,2015-08-11,13.417963
43571,S02000010_20160825,397,NaN,NaN,2.518892,269.521411,NaN,2.518892,NaN,NaN,...,ALSACE,62,246.0,None,118.387909,NaN,NaN,None,2016-08-25,14.207038
43572,S02000010_20170703,410,NaN,NaN,NaN,290.243902,NaN,NaN,NaN,NaN,...,ALSACE,62,246.0,None,51.219512,NaN,NaN,None,2017-07-03,15.749292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49226,S06940940_20100708,438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,JURA-PREALPES DU NORD,2,256.0,P,2.283105,NaN,NaN,None,2010-07-08,16.094491
49227,S06940940_20230623,408,NaN,NaN,NaN,NaN,NaN,4.901961,NaN,NaN,...,JURA-PREALPES DU NORD,2,256.0,P,19.607843,NaN,NaN,None,2023-06-23,15.392658
49228,S06960950_20160629,401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,JURA-PREALPES DU NORD,2,366.0,TP,24.937656,NaN,NaN,None,2016-06-29,19.942872
49229,S06960950_20180719,416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,JURA-PREALPES DU NORD,2,366.0,TP,33.653846,NaN,NaN,None,2018-07-19,20.084038


In [21]:
scored_df.to_parquet('results/0013/013_Catboost_FullIBD_Training_pred.parquet')

In [26]:
model2, metrics2, scored_df2 = train_catboost_region_tunueado(
    full,
    base = base_full2
)

CatBoostError: catboost/private/libs/options/catboost_options.cpp:637: Error: rsm on GPU is supported for pairwise modes only